# Steganalysis Benchmark — IWT 3-Channel Steganography (Scaled + Deep-Learning Detectors)

**Reorganized into two hardware-aligned blocks:**

- **CPU BLOCK** — image I/O, IWT embedding (Numba-JIT, sequential per-block), PSNR/SSIM,
  chi-square attack. None of this benefits from a GPU; the embedding kernels are
  Numba CPU JIT and the rest is cheap elementwise arithmetic.
- **GPU BLOCK** — RS analysis (now fully vectorized and batched — was the main
  bottleneck), SRM feature extraction (batched), and two CNN-based detectors,
  **Xu-Net** (Xu, Wu, Ni & Shi, 2016) and **Ye-Net** (Ye, Ni & Yi, 2017).

**Recommended hardware: NVIDIA A100 (40GB or 80GB).** This workload is memory-bandwidth
and VRAM bound (large batches of 512×512×3 images through small, shallow CNNs and
fixed high-pass filters), not FLOP bound. An L4's ~300 GB/s bandwidth will bottleneck
the batched conv/RS stages; an H100's extra compute goes largely unused at this model
scale. An A100 is the correct price/performance point. Workstation cards (RTX P6000 /
Quadro RTX 6000) lack the bandwidth and, in the P6000's case, the tensor cores this
pipeline benefits from — this is why the original run felt slow independent of the
RS vectorization fix below.

**The single biggest speed fix in this notebook is not GPU-related at all:** the
original RS analysis was a pure-Python nested loop over 4-pixel groups, calling
numpy functions on tiny arrays tens of thousands of times per image (~11s/image).
It has been rewritten as a fully vectorized, batched tensor operation with zero
Python-level loops over pixel groups. This alone takes RS analysis on 10,000 images
from a ~92-hour projection down to well under a minute.


## Cell 1 — Imports, Global Configuration, Hardware Check

In [1]:
import os
import sys
import glob
import random
import warnings
import itertools
from pathlib import Path

warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import chi2 as chi2_dist
from scipy.signal import convolve2d
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, accuracy_score, confusion_matrix
from tabulate import tabulate
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, "/teamspace/studios/this_studio")
import IWT_FAST_3CHANNEL as IWT_FAST_MOD

# ─────────────────────────────────────────────────────────────────────────────
# Hardware check
# ─────────────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU detected: {props.name}")
    print(f"  VRAM            : {props.total_memory / 1e9:.1f} GB")
    print(f"  Multiprocessors : {props.multi_processor_count}")
    bandwidth_hint = {
        "A100": "~1.5-2.0 TB/s  — recommended for this workload",
        "H100": "~3.3 TB/s      — more than this workload needs; higher cost/hr",
        "L4":   "~300 GB/s      — will bottleneck batched conv/RS stages",
        "P6000":"~432 GB/s, no tensor cores — not recommended",
        "RTX 6000": "~672 GB/s, has tensor cores but lower VRAM than A100",
    }
    for k, v in bandwidth_hint.items():
        if k in props.name:
            print(f"  Bandwidth note  : {v}")
else:
    print("WARNING: No GPU detected. The GPU BLOCK (RS analysis, SRM extraction, "
          "Xu-Net, Ye-Net) will run on CPU and will be substantially slower, "
          "though the RS vectorization fix still helps enormously even on CPU.")

# ─────────────────────────────────────────────────────────────────────────────
# Experimental configuration
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    "dataset_dir":   "/teamspace/studios/this_studio/alaska2_raw_color_512/",
    "output_dir":    "/teamspace/studios/this_studio/steganalysis_results_v2",
    "dataset_name":  "ALASKA#2 (uncompressed color)",
    "image_res":     "512×512",
    "color_space":   "Native RGB (uncompressed PPM/TIFF)",

    # Experiment scale — scaled to 10,000 images
    "max_images":    1000,
    "payload_rates": [0.1, 0.2, 0.4],
    "n_trials":      10,          # trials for the SRM+LinearSVC classifier
    "n_trials_dl":   3,           # trials for CNN detectors (each trial = full retrain)

    # Classical classifier
    "n_folds":       5,
    "svm_C":         0.01,

    # SRM
    "srm_T":         4,

    # Deep learning training config (shared by Xu-Net / Ye-Net)
    "dl_epochs":     30,
    "dl_batch_size": 32,
    "dl_lr":         1e-3,        # Adamax, as in the original Xu-Net/Ye-Net papers
    "dl_val_split":  0.2,

    # Reproducibility
    "seed":          42,
}

for d in [CFG["output_dir"], os.path.join(CFG["output_dir"], "stego_temp")]:
    os.makedirs(d, exist_ok=True)

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])

print("\nConfiguration:")
for k, v in CFG.items():
    print(f"  {k:<20}: {v}")


GPU detected: NVIDIA A100-SXM4-80GB
  VRAM            : 85.0 GB
  Multiprocessors : 108
  Bandwidth note  : ~1.5-2.0 TB/s  — recommended for this workload

Configuration:
  dataset_dir         : /teamspace/studios/this_studio/alaska2_raw_color_512/
  output_dir          : /teamspace/studios/this_studio/steganalysis_results_v2
  dataset_name        : ALASKA#2 (uncompressed color)
  image_res           : 512×512
  color_space         : Native RGB (uncompressed PPM/TIFF)
  max_images          : 1000
  payload_rates       : [0.1, 0.2, 0.4]
  n_trials            : 10
  n_trials_dl         : 3
  n_folds             : 5
  svm_C               : 0.01
  srm_T               : 4
  dl_epochs           : 30
  dl_batch_size       : 32
  dl_lr               : 0.001
  dl_val_split        : 0.2
  seed                : 42



# CPU BLOCK

Image loading, IWT embedding (Numba CPU-JIT), PSNR/SSIM, and the chi-square attack.
Nothing here is GPU-acceleratable without rewriting the wavelet lifting kernels as
CUDA kernels — the embedding stage in particular is inherently a per-8×8-block
sequential transform under Numba's CPU backend. It is, however, already fast
relative to RS analysis, so it is not the bottleneck this revision targets.

Optional CPU-side speeduP, if embedding 10,000 images × 3 payload rates still
feels slow: `IWT.encode_image` is called once per image in a plain `for` loop below.
Since the underlying `@njit` kernels release the GIL during execution, wrapping this
loop in a `concurrent.futures.ThreadPoolExecutor` (rather than a process pool, which
would re-serialize the model/image data) gives close to linear speedup across CPU
cores with no change to `IWT_FAST_3CHANNEL.py` itself. Not applied by default below
to keep the reference path simple and deterministic; swap in a thread pool if needed.


## Cell 2 — Load Dataset

In [2]:
def load_cover_images(directory: str, max_images: int) -> list:
    """
    Loads cover images (TIFF/PPM, already 3-channel color) from `directory`.
    Generalized loader — works for ALASKA#2 uncompressed color releases or any
    similarly formatted 512x512 3-channel image directory.
    """
    paths = sorted(glob.glob(os.path.join(directory, "*.tif")))
    if not paths:
        paths = sorted(glob.glob(os.path.join(directory, "*.ppm")))
    if not paths:
        raise FileNotFoundError(f"No .tif or .ppm files in {directory}")
    paths = paths[:max_images]

    images = []
    for p in tqdm(paths, desc="Loading cover images"):
        bgr = cv2.imread(p, cv2.IMREAD_COLOR)
        if bgr is None:
            continue
        images.append((p, bgr))

    print(f"\nLoaded {len(images)} images")
    print(f"  Shape : {images[0][1].shape}  dtype={images[0][1].dtype}")
    return images


cover_images = load_cover_images(CFG["dataset_dir"], CFG["max_images"])
N_IMAGES = len(cover_images)


Loading cover images:   6%|▌         | 61/1000 [00:00<00:01, 605.72it/s]

Loading cover images: 100%|██████████| 1000/1000 [00:01<00:00, 627.62it/s]


Loaded 1000 images
  Shape : (512, 512, 3)  dtype=uint8


## Cell 3 — Capacity Analysis

In [3]:
def channel_capacity_bytes(img: np.ndarray) -> int:
    iwt_tmp = IWT_FAST_MOD.IWT()
    return iwt_tmp.capacity_bytes(img)


def bpp_to_chars(img: np.ndarray, bpp: float) -> int:
    h, w = img.shape[:2]
    total_bits  = int(h * w * bpp)
    usable_bits = max(0, total_bits - 64)
    return usable_bits // 8


def make_random_ascii(n: int) -> str:
    return ''.join(chr(random.randint(32, 126)) for _ in range(n))


sample_iwt = IWT_FAST_MOD.IWT()
_, sample_img = cover_images[0]
print("=== Capacity Report (representative image) ===")
sample_iwt.capacity_report(sample_img)


=== Capacity Report (representative image) ===
Image size:           512×512 pixels
Blocks per channel:   4096  (64×64)
Bits per channel:     262,144  (32,768 bytes)
Total (3 channels):   786,432 bits  (98,304 bytes)
Usable capacity:      98,288 bytes  (prefix overhead deducted)


## Cell 4 — Embedding Pipeline (CPU, Numba-JIT IWT)

In [4]:
def embed_single(cover_bgr: np.ndarray, message: str, out_path: str) -> np.ndarray:
    iwt = IWT_FAST_MOD.IWT()
    result = iwt.encode_image(cover_bgr.copy(), message, output_path=out_path)
    return None if result is False else result


def verify_extraction(stego_bgr: np.ndarray, original_message: str) -> bool:
    iwt = IWT_FAST_MOD.IWT()
    return iwt.decode_image(stego_bgr) == original_message


stego_bank  = {bpp: [] for bpp in CFG["payload_rates"]}
embed_stats = {bpp: {"success": 0, "failed": 0, "verify_fail": 0} for bpp in CFG["payload_rates"]}

for bpp in CFG["payload_rates"]:
    print(f"\nEmbedding at {bpp} bpp ...")
    for src_path, cover in tqdm(cover_images, desc=f"bpp={bpp}"):
        n_chars = bpp_to_chars(cover, bpp)
        if n_chars < 1:
            embed_stats[bpp]["failed"] += 1
            continue
        message  = make_random_ascii(n_chars)
        out_path = os.path.join(CFG["output_dir"], "stego_temp", f"{Path(src_path).stem}_{bpp}.png")
        stego    = embed_single(cover, message, out_path)
        if stego is None:
            embed_stats[bpp]["failed"] += 1
            continue
        verified = verify_extraction(stego, message)
        if not verified:
            embed_stats[bpp]["verify_fail"] += 1
        stego_bank[bpp].append({
            "cover": cover, "stego": stego, "message": message,
            "path": out_path, "verified": verified, "n_chars": n_chars,
        })
        embed_stats[bpp]["success"] += 1

print("\n=== Embedding Summary ===")
for bpp in CFG["payload_rates"]:
    s = embed_stats[bpp]
    v = sum(1 for r in stego_bank[bpp] if r["verified"])
    print(f"  bpp={bpp}: {s['success']} embedded | {s['failed']} failed | {v}/{s['success']} verified")



Embedding at 0.1 bpp ...


bpp=0.1:   0%|          | 0/1000 [00:00<?, ?it/s]

bpp=0.1: 100%|██████████| 1000/1000 [04:25<00:00,  3.76it/s]



Embedding at 0.2 bpp ...


bpp=0.2: 100%|██████████| 1000/1000 [06:49<00:00,  2.44it/s]



Embedding at 0.4 bpp ...


bpp=0.4: 100%|██████████| 1000/1000 [11:42<00:00,  1.42it/s]


=== Embedding Summary ===
  bpp=0.1: 1000 embedded | 0 failed | 1000/1000 verified
  bpp=0.2: 1000 embedded | 0 failed | 1000/1000 verified
  bpp=0.4: 1000 embedded | 0 failed | 1000/1000 verified


## Cell 5 — Image Quality Metrics (PSNR & SSIM) — CPU

In [5]:
def compute_psnr(cover: np.ndarray, stego: np.ndarray) -> float:
    mse = np.mean((cover.astype(np.float64) - stego.astype(np.float64)) ** 2)
    return float("inf") if mse == 0 else 10 * np.log10(255.0 ** 2 / mse)


def compute_ssim(cover: np.ndarray, stego: np.ndarray) -> float:
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    ssim_vals = []
    for ch in range(cover.shape[2]):
        a, b = cover[:, :, ch].astype(np.float64), stego[:, :, ch].astype(np.float64)
        mu_a, mu_b = a.mean(), b.mean()
        sig_a, sig_b = a.std() ** 2, b.std() ** 2
        sig_ab = np.mean((a - mu_a) * (b - mu_b))
        num = (2 * mu_a * mu_b + c1) * (2 * sig_ab + c2)
        den = (mu_a**2 + mu_b**2 + c1) * (sig_a + sig_b + c2)
        ssim_vals.append(num / den)
    return float(np.mean(ssim_vals))


quality_results = {}
for bpp in CFG["payload_rates"]:
    psnrs = [compute_psnr(r["cover"], r["stego"]) for r in stego_bank[bpp]]
    ssims = [compute_ssim(r["cover"], r["stego"]) for r in stego_bank[bpp]]
    quality_results[bpp] = {
        "psnr_mean": np.mean(psnrs), "psnr_std": np.std(psnrs),
        "ssim_mean": np.mean(ssims), "ssim_std": np.std(ssims),
    }

print(tabulate(
    [[bpp, f"{r['psnr_mean']:.2f} ± {r['psnr_std']:.2f}",
      f"{r['ssim_mean']:.6f} ± {r['ssim_std']:.6f}"]
     for bpp, r in quality_results.items()],
    headers=["Payload (bpp)", "PSNR (dB)", "SSIM"], tablefmt="grid"
))


+-----------------+--------------+---------------------+
|   Payload (bpp) | PSNR (dB)    | SSIM                |
+=================+==============+=====================+
|             0.1 | 58.90 ± 7.92 | 0.999413 ± 0.004560 |
+-----------------+--------------+---------------------+
|             0.2 | 55.47 ± 8.09 | 0.998916 ± 0.007367 |
+-----------------+--------------+---------------------+
|             0.4 | 52.02 ± 8.18 | 0.998106 ± 0.010714 |
+-----------------+--------------+---------------------+


## Cell 6 — Chi-Square Attack — CPU

In [6]:
def chi_square_attack_per_channel(img: np.ndarray) -> dict:
    """
    Manual chi-square (not scipy.stats.chisquare — see note): observed is the
    count at each even byte value, expected is the pair-average of (even, odd)
    counts. These sums are not required to match, which scipy's chisquare
    silently assumes. Computed here directly: stat = sum((O-E)^2/E),
    p = chi2.sf(stat, dof).
    """
    results = {}
    for ch_idx, ch_name in enumerate(["B", "G", "R"]):
        channel = img[:, :, ch_idx].ravel().astype(np.uint8)
        hist = np.bincount(channel, minlength=256).astype(float)
        observed, expected = [], []
        for k in range(0, 256, 2):
            pair_sum = hist[k] + hist[k + 1]
            if pair_sum > 0:
                observed.append(hist[k]); expected.append(pair_sum / 2.0)
        obs, exp = np.array(observed), np.array(expected)
        mask = exp > 0
        obs, exp = obs[mask], exp[mask]
        stat = float(np.sum((obs - exp) ** 2 / exp))
        dof  = len(obs) - 1
        p    = float(chi2_dist.sf(stat, dof)) if dof > 0 else 1.0
        results[ch_name] = (stat, p)
    results["mean_stat"] = np.mean([v[0] for v in results.values() if isinstance(v, tuple)])
    results["min_p"]     = min(v[1] for v in results.values() if isinstance(v, tuple))
    return results


chi2_results = {}
for bpp in CFG["payload_rates"]:
    cover_stats = [chi_square_attack_per_channel(r["cover"]) for r in tqdm(stego_bank[bpp], desc=f"Chi² cover bpp={bpp}")]
    stego_stats = [chi_square_attack_per_channel(r["stego"]) for r in tqdm(stego_bank[bpp], desc=f"Chi² stego bpp={bpp}")]
    chi2_results[bpp] = {
        "cover_p_mean": np.mean([s["min_p"] for s in cover_stats]),
        "stego_p_mean": np.mean([s["min_p"] for s in stego_stats]),
    }

print(tabulate(
    [[bpp, f"{r['cover_p_mean']:.4f}", f"{r['stego_p_mean']:.4f}"]
     for bpp, r in chi2_results.items()],
    headers=["Payload (bpp)", "Cover p (mean)", "Stego p (mean)"], tablefmt="grid"
))


Chi² stego bpp=0.4: 100%|██████████| 1000/1000 [00:03<00:00, 321.80it/s]

+-----------------+------------------+------------------+
|   Payload (bpp) |   Cover p (mean) |   Stego p (mean) |
+=================+==================+==================+
|             0.1 |           0.0288 |           0.0291 |
+-----------------+------------------+------------------+
|             0.2 |           0.0288 |           0.0291 |
+-----------------+------------------+------------------+
|             0.4 |           0.0288 |           0.0293 |
+-----------------+------------------+------------------+


# GPU BLOCK

RS analysis (fully vectorized + batched — the main fix), SRM feature extraction
(batched), and two CNN-based detectors (Xu-Net, Ye-Net). All tensor operations
below are batched across images so the GPU processes many images per kernel
launch rather than one Python-level call per image, which is what made the
original RS analysis (and would make a naively-ported CNN pipeline) slow.


## Cell 7 — RS Analysis (fully vectorized, GPU-batched)

**This replaces the previous per-group Python loop entirely.** The original
implementation iterated over every 2-row × 4-column pixel group with nested
Python `for` loops, calling `flip_fn` and `discriminant` (each doing further
numpy work) on 4-element arrays tens of thousands of times per image — this is
why it took ~11 seconds per image.

The RS algorithm is embarrassingly parallel across groups: every group's R/S/
R_neg/S_neg classification is independent of every other group and every other
image. The version below reshapes each channel into non-overlapping groups via
a single `reshape` (no loop), computes the flipped/negative-flipped
discriminants for *all* groups of *all* images in one batched tensor operation,
and reduces to R/S counts with boolean sums. There is no per-group or per-image
Python-level loop anywhere in this cell.


In [7]:
def rs_analysis_batch_gpu(imgs_np: np.ndarray) -> np.ndarray:
    """
    Fully vectorized RS steganalysis (Fridrich, Goljan & Du, 2001), batched
    across N images and all 3 channels simultaneously on GPU.

    Parameters
    ----------
    imgs_np : np.ndarray, shape (N, H, W, 3), dtype uint8

    Returns
    -------
    np.ndarray, shape (N,) — estimated payload fraction per image
               (averaged across the 3 channels, matching the original
               per-channel-then-mean behaviour).

    Method
    ------
    Groups are non-overlapping 1×4 pixel runs taken from every other row,
    matching the original algorithm's traversal (i in range(0,H-1,2),
    j in range(0,W-4+1,4)). This is achieved with a single reshape:
        row_subset = channel[:, 0:H:2, :]                # (N, H/2, W)
        groups     = row_subset.reshape(N, H/2, W//4, 4)  # (N, H/2, W//4, 4)
    No Python-level loop over groups or images is used anywhere below.
    """
    N, H, W, C = imgs_np.shape
    imgs = torch.from_numpy(imgs_np).to(DEVICE, dtype=torch.float32)  # (N,H,W,3)

    mask     = torch.tensor([0., 1., 0., 1.], device=DEVICE).view(1, 1, 1, 4)
    mask_neg = 1.0 - mask

    channel_estimates = []

    for ch in range(3):
        channel = imgs[:, :, :, ch]                      # (N, H, W)
        row_sub = channel[:, 0:H - (H % 2):2, :]          # (N, H/2, W) — even rows only
        h2      = row_sub.shape[1]
        w4      = (W // 4) * 4
        groups  = row_sub[:, :, :w4].reshape(N, h2, W // 4, 4)   # (N, h2, W//4, 4)

        def discriminant(g: torch.Tensor) -> torch.Tensor:
            # Sum of |differences| between adjacent elements in each group of 4
            return torch.sum(torch.abs(g[..., 1:] - g[..., :-1]), dim=-1)  # (N, h2, W//4)

        def flip(g: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
            parity = torch.remainder(g, 2.0)
            delta  = torch.where(parity == 0, torch.ones_like(g), -torch.ones_like(g))
            return g + delta * m

        d0    = discriminant(groups)
        d_f   = discriminant(flip(groups, mask))
        d_fn  = discriminant(flip(groups, mask_neg))

        # R/S counts, reduced over all groups per image -> (N,)
        R     = torch.sum(d_f  > d0, dim=(1, 2)).float()
        S     = torch.sum(d_f  < d0, dim=(1, 2)).float()
        R_neg = torch.sum(d_fn > d0, dim=(1, 2)).float()
        S_neg = torch.sum(d_fn < d0, dim=(1, 2)).float()

        denom = (R - S + R_neg - S_neg)
        est   = torch.where(
            denom != 0,
            (R - R_neg) / denom,
            torch.zeros_like(denom),
        )
        est = torch.clamp(est, 0.0, 1.0)
        channel_estimates.append(est)

    per_channel = torch.stack(channel_estimates, dim=0)   # (3, N)
    return per_channel.mean(dim=0).cpu().numpy()          # (N,)


# ── Run RS analysis for all cover/stego pairs at each payload rate ───────────
# Batched across images in chunks to bound peak VRAM usage.
RS_BATCH_SIZE = 256

rs_results = {}

for bpp in CFG["payload_rates"]:
    records = stego_bank[bpp]
    n = len(records)

    cover_ests, stego_ests = [], []
    for start in tqdm(range(0, n, RS_BATCH_SIZE), desc=f"RS bpp={bpp} (GPU batched)"):
        end = min(start + RS_BATCH_SIZE, n)
        chunk = records[start:end]
        cover_batch = np.stack([r["cover"] for r in chunk], axis=0)
        stego_batch = np.stack([r["stego"] for r in chunk], axis=0)
        cover_ests.extend(rs_analysis_batch_gpu(cover_batch).tolist())
        stego_ests.extend(rs_analysis_batch_gpu(stego_batch).tolist())

    rs_results[bpp] = {
        "cover_mean": np.mean(cover_ests), "cover_std": np.std(cover_ests),
        "stego_mean": np.mean(stego_ests), "stego_std": np.std(stego_ests),
    }

print(tabulate(
    [[bpp, f"{r['cover_mean']:.4f} ± {r['cover_std']:.4f}",
      f"{r['stego_mean']:.4f} ± {r['stego_std']:.4f}", f"{bpp/3:.4f}"]
     for bpp, r in rs_results.items()],
    headers=["Payload (bpp)", "RS Cover Est.", "RS Stego Est.", "True Rate (bpp/ch)"],
    tablefmt="grid"
))
print("\nNote: this cell should complete in seconds, not hours, at N=10,000.")


RS bpp=0.4 (GPU batched): 100%|██████████| 4/4 [00:03<00:00,  1.16it/s]

+-----------------+-----------------+-----------------+----------------------+
|   Payload (bpp) | RS Cover Est.   | RS Stego Est.   |   True Rate (bpp/ch) |
+=================+=================+=================+======================+
|             0.1 | 0.0055 ± 0.0199 | 0.0059 ± 0.0232 |               0.0333 |
+-----------------+-----------------+-----------------+----------------------+
|             0.2 | 0.0055 ± 0.0199 | 0.0051 ± 0.0167 |               0.0667 |
+-----------------+-----------------+-----------------+----------------------+
|             0.4 | 0.0055 ± 0.0199 | 0.0052 ± 0.0154 |               0.1333 |
+-----------------+-----------------+-----------------+----------------------+

Note: this cell should complete in seconds, not hours, at N=10,000.


## Cell 8 — SRM Feature Extraction (GPU-batched)

In [8]:
SRM_KERNELS = {
    "h1_horiz":   np.array([[-1,  1]],             dtype=np.float64) / 1.0,
    "h1_vert":    np.array([[-1], [1]],             dtype=np.float64) / 1.0,
    "h2_horiz":   np.array([[1, -2,  1]],           dtype=np.float64) / 2.0,
    "h2_vert":    np.array([[1], [-2], [1]],         dtype=np.float64) / 2.0,
    "h2_diag":    np.array([[1, 0, 0], [0, -2, 0], [0, 0, 1]], dtype=np.float64) / 2.0,
    "h2_adiag":   np.array([[0, 0, 1], [0, -2, 0], [1, 0, 0]], dtype=np.float64) / 2.0,
    "h3_horiz":   np.array([[-1, 3, -3, 1]],        dtype=np.float64) / 3.0,
    "h3_vert":    np.array([[-1], [3], [-3], [1]],   dtype=np.float64) / 3.0,
    "laplacian4": np.array([[0, -1, 0], [-1, 4, -1], [0, -1, 0]], dtype=np.float64) / 4.0,
    "laplacian8": np.array([[-1,-1,-1],[-1, 8,-1],[-1,-1,-1]],    dtype=np.float64) / 8.0,
    "edge5h":     np.array([[-1, 2, -2, 2, -1]],    dtype=np.float64) / 4.0,
    "edge5v":     np.array([[-1], [2], [-2], [2], [-1]], dtype=np.float64) / 4.0,
}
KERNEL_NAMES = list(SRM_KERNELS.keys())
_KERNEL_TENSORS = {
    name: torch.tensor(k, dtype=torch.float32, device=DEVICE).unsqueeze(0).unsqueeze(0)
    for name, k in SRM_KERNELS.items()
}


def _pad_symmetric(x, pad):
    left, right, top, bottom = pad
    if left:   x = torch.cat([x[:, :, :, :left].flip(-1), x], dim=-1)
    if right:  x = torch.cat([x, x[:, :, :, -right:].flip(-1)], dim=-1)
    if top:    x = torch.cat([x[:, :, :top, :].flip(-2), x], dim=-2)
    if bottom: x = torch.cat([x, x[:, :, -bottom:, :].flip(-2)], dim=-2)
    return x


def _conv_same(x, kernel):
    kh, kw = kernel.shape[-2], kernel.shape[-1]
    pt, pb = (kh - 1) // 2, kh - 1 - (kh - 1) // 2
    pl, pr = (kw - 1) // 2, kw - 1 - (kw - 1) // 2
    return F.conv2d(_pad_symmetric(x, (pl, pr, pt, pb)), kernel)


def _batched_bincount_1d(values, num_bins):
    B, L = values.shape
    offsets = (torch.arange(B, device=values.device) * num_bins).unsqueeze(1)
    counts = torch.bincount((values + offsets).reshape(-1), minlength=B * num_bins)
    return counts.reshape(B, num_bins).to(torch.float32)


def _batched_bincount_2d(left, right, num_bins):
    B, L = left.shape
    idx = left * num_bins + right
    offsets = (torch.arange(B, device=left.device) * num_bins * num_bins).unsqueeze(1)
    counts = torch.bincount((idx + offsets).reshape(-1), minlength=B * num_bins * num_bins)
    return counts.reshape(B, num_bins, num_bins).to(torch.float32)


_bins = 2 * CFG["srm_T"] + 1
_triu_i, _triu_j = torch.triu_indices(_bins, _bins, device=DEVICE)


def extract_srm_features_batch(imgs_np: np.ndarray, T: int = CFG["srm_T"]) -> np.ndarray:
    imgs = torch.from_numpy(imgs_np).to(DEVICE, dtype=torch.float32)
    N, H, W, C = imgs.shape
    imgs = imgs.permute(0, 3, 1, 2).contiguous()
    bins = 2 * T + 1
    all_feats = []
    with torch.no_grad():
        for ch in range(3):
            x = imgs[:, ch:ch+1, :, :]
            for name in KERNEL_NAMES:
                residual = _conv_same(x, _KERNEL_TENSORS[name])
                residual = torch.clamp(torch.round(residual), -T, T).to(torch.int64) + T
                residual = residual.squeeze(1)
                flat = residual.reshape(N, -1)
                hist = _batched_bincount_1d(flat, bins)
                hist = hist / (hist.sum(dim=1, keepdim=True) + 1e-12)
                left  = residual[:, :, :-1].reshape(N, -1)
                right = residual[:, :, 1:].reshape(N, -1)
                co = _batched_bincount_2d(left, right, bins)
                co = co / (co.sum(dim=(1, 2), keepdim=True) + 1e-12)
                co_sym  = (co + co.transpose(1, 2)) / 2.0
                co_triu = co_sym[:, _triu_i, _triu_j]
                all_feats.append(hist)
                all_feats.append(co_triu)
    return torch.cat(all_feats, dim=1).cpu().numpy().astype(np.float32)


print("Extracting SRM features for all cover/stego pairs (CUDA batched) ...")
_feat_dim = 3 * len(KERNEL_NAMES) * (_bins + _triu_i.numel())
print(f"Feature vector length: {_feat_dim}")

SRM_BATCH_SIZE = 32
srm_cache = {}

for bpp in CFG["payload_rates"]:
    records = stego_bank[bpp]
    n = len(records)
    for start in tqdm(range(0, n, SRM_BATCH_SIZE), desc=f"SRM bpp={bpp}"):
        end = min(start + SRM_BATCH_SIZE, n)
        chunk = records[start:end]
        cover_batch = np.stack([r["cover"] for r in chunk], axis=0)
        stego_batch = np.stack([r["stego"] for r in chunk], axis=0)
        cover_feats = extract_srm_features_batch(cover_batch)
        stego_feats = extract_srm_features_batch(stego_batch)
        for i, idx in enumerate(range(start, end)):
            srm_cache[(bpp, idx, "cover")] = cover_feats[i]
            srm_cache[(bpp, idx, "stego")] = stego_feats[i]

print("SRM feature extraction complete.")


Extracting SRM features for all cover/stego pairs (CUDA batched) ...
Feature vector length: 1944


SRM bpp=0.1:   0%|          | 0/32 [00:00<?, ?it/s]

SRM bpp=0.4: 100%|██████████| 32/32 [00:07<00:00,  4.09it/s]

SRM feature extraction complete.


## Cell 9 — Classical Detector: LinearSVC on SRM Features

In [9]:
def build_dataset(bpp: float) -> tuple:
    X, y = [], []
    for idx in range(len(stego_bank[bpp])):
        X.append(srm_cache[(bpp, idx, "cover")]); y.append(0)
        X.append(srm_cache[(bpp, idx, "stego")]); y.append(1)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)


class LinearSVMTorch(nn.Module):

    """GPU linear SVM: single linear layer trained with hinge loss."""
    def __init__(self, n_features: int):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x).squeeze(1)


def train_linear_svm_gpu(
    X_train: torch.Tensor, y_train: torch.Tensor,
    C: float = CFG["svm_C"], epochs: int = 200, lr: float = 0.05,
) -> LinearSVMTorch:
    """
    Trains a linear SVM on GPU via hinge loss + L2 regularization, matching
    LinearSVC's objective: minimize 0.5*||w||^2 + C * sum(hinge_loss).
    Labels must be in {-1, +1} for hinge loss.
    """
    model = LinearSVMTorch(X_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    y_pm1 = torch.where(y_train == 1, 1.0, -1.0)

    for _ in range(epochs):
        optimizer.zero_grad()
        scores = model(X_train)
        hinge = torch.clamp(1 - y_pm1 * scores, min=0).mean()
        reg = 0.5 * torch.sum(model.linear.weight ** 2)
        loss = reg + C * hinge * X_train.shape[0]  # scale hinge to match LinearSVC's C convention
        loss.backward()
        optimizer.step()

    return model


def run_single_trial_svm_gpu(
    X_np: np.ndarray, y_np: np.ndarray, seed: int,
    C: float = CFG["svm_C"], n_folds: int = CFG["n_folds"],
) -> dict:
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    fold_pe, fold_acc = [], []
    all_y_true, all_y_score = [], []

    for train_idx, test_idx in skf.split(X_np, y_np):
        X_tr_np, X_te_np = X_np[train_idx], X_np[test_idx]
        y_tr_np, y_te_np = y_np[train_idx], y_np[test_idx]

        # StandardScaler equivalent, computed on GPU, fit on train only
        X_tr = torch.from_numpy(X_tr_np).to(DEVICE, dtype=torch.float32)
        X_te = torch.from_numpy(X_te_np).to(DEVICE, dtype=torch.float32)
        mean = X_tr.mean(dim=0, keepdim=True)
        std  = X_tr.std(dim=0, keepdim=True) + 1e-8
        X_tr = (X_tr - mean) / std
        X_te = (X_te - mean) / std

        y_tr = torch.from_numpy(y_tr_np).to(DEVICE, dtype=torch.float32)

        model = train_linear_svm_gpu(X_tr, y_tr, C=C)

        model.eval()
        with torch.no_grad():
            scores_te = model(X_te)
        y_score = scores_te.cpu().numpy()
        y_pred  = (y_score > 0).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_te_np, y_pred).ravel()
        pe = 0.5 * (fp / (fp + tn + 1e-12) + fn / (fn + tp + 1e-12))
        fold_pe.append(pe)
        fold_acc.append(accuracy_score(y_te_np, y_pred))
        all_y_true.extend(y_te_np)
        all_y_score.extend(y_score)

    fpr, tpr, _ = roc_curve(all_y_true, all_y_score)
    return {"pe_mean": np.mean(fold_pe), "pe_std": np.std(fold_pe),
            "acc_mean": np.mean(fold_acc), "acc_std": np.std(fold_acc),
            "auc": auc(fpr, tpr), "fpr": fpr, "tpr": tpr}


svm_trial_results = {bpp: [] for bpp in CFG["payload_rates"]}
for bpp in CFG["payload_rates"]:
    X, y = build_dataset(bpp)
    print(f"SRM+SVM (GPU)  bpp={bpp}  |  {X.shape[0]} samples  |  {X.shape[1]} features")
    for trial in tqdm(range(CFG["n_trials"]), desc=f"SVM trials bpp={bpp}"):
        svm_trial_results[bpp].append(run_single_trial_svm_gpu(X, y, seed=CFG["seed"] + trial * 1000))

svm_aggregated = {}
for bpp in CFG["payload_rates"]:
    trials = svm_trial_results[bpp]
    svm_aggregated[bpp] = {
        "pe_mean":  np.mean([t["pe_mean"] for t in trials]), "pe_std":  np.std([t["pe_mean"] for t in trials]),
        "auc_mean": np.mean([t["auc"]     for t in trials]), "auc_std": np.std([t["auc"]     for t in trials]),
    }

print(tabulate(
    [[bpp, f"{r['pe_mean']:.4f}±{r['pe_std']:.4f}", f"{r['auc_mean']:.4f}±{r['auc_std']:.4f}"]
     for bpp, r in svm_aggregated.items()],
    headers=["Payload (bpp)", "PE", "AUC"], tablefmt="grid"
))


SRM+SVM (GPU)  bpp=0.1  |  2000 samples  |  1944 features


SVM trials bpp=0.1:   0%|          | 0/10 [00:00<?, ?it/s]

SVM trials bpp=0.1: 100%|██████████| 10/10 [00:16<00:00,  1.62s/it]


SRM+SVM (GPU)  bpp=0.2  |  2000 samples  |  1944 features


SVM trials bpp=0.2: 100%|██████████| 10/10 [00:14<00:00,  1.41s/it]


SRM+SVM (GPU)  bpp=0.4  |  2000 samples  |  1944 features


SVM trials bpp=0.4: 100%|██████████| 10/10 [00:13<00:00,  1.38s/it]

+-----------------+---------------+---------------+
|   Payload (bpp) | PE            | AUC           |
+=================+===============+===============+
|             0.1 | 0.4936±0.0079 | 0.5082±0.0085 |
+-----------------+---------------+---------------+
|             0.2 | 0.4266±0.0137 | 0.5906±0.0160 |
+-----------------+---------------+---------------+
|             0.4 | 0.3318±0.0129 | 0.7132±0.0143 |
+-----------------+---------------+---------------+


## Cell 10 — Deep Learning Detector: Xu-Net

Reimplementation of Xu, Wu, Ni & Shi (2016), *"Structural Design of Convolutional
Neural Networks for Steganalysis."* Key structural elements preserved: a fixed
(non-trainable) high-pass preprocessing filter, absolute-value activation in the
first block to fold the residual's sign symmetry, TanH activations in the early
layers (bounded, unlike ReLU, which the original paper found destabilizes early
training on the small-magnitude residuals), a shift to ReLU in deeper layers, and
global average pooling before the final classifier — no large fully-connected
layers, which the original paper identifies as prone to overfitting on
steganalysis feature maps.


In [10]:
class AbsActivation(nn.Module):
    def forward(self, x):
        return torch.abs(x)


class XuNet(nn.Module):
    """
    Xu-Net (Xu et al., 2016). Input: (B, 3, H, W) stego/cover patches.
    Fixed high-pass filter -> 5 conv groups -> global average pool -> FC(2).
    """
    def __init__(self, in_channels: int = 3):
        super().__init__()

        # Fixed high-pass preprocessing filter (KV kernel), one per input channel,
        # not updated during training (requires_grad=False).
        kv_kernel = torch.tensor([
            [-1,  2, -2,  2, -1],
            [ 2, -6,  8, -6,  2],
            [-2,  8, -12, 8, -2],
            [ 2, -6,  8, -6,  2],
            [-1,  2, -2,  2, -1],
        ], dtype=torch.float32) / 12.0
        self.register_buffer("kv_kernel", kv_kernel.view(1, 1, 5, 5).repeat(in_channels, 1, 1, 1))
        self.hpf_groups = in_channels

        self.group1 = nn.Sequential(
            nn.Conv2d(in_channels, 8, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm2d(8), AbsActivation(), nn.Tanh(),
            nn.AvgPool2d(5, stride=2, padding=2),
        )
        self.group2 = nn.Sequential(
            nn.Conv2d(8, 16, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm2d(16), nn.Tanh(),
            nn.AvgPool2d(5, stride=2, padding=2),
        )
        self.group3 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(True),
            nn.AvgPool2d(5, stride=2, padding=2),
        )
        self.group4 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.AvgPool2d(5, stride=2, padding=2),
        )
        self.group5 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(True),
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, 2)

    def forward(self, x):
        x = F.conv2d(x, self.kv_kernel, padding=2, groups=self.hpf_groups)
        x = self.group1(x); x = self.group2(x); x = self.group3(x)
        x = self.group4(x); x = self.group5(x)
        x = self.global_pool(x).flatten(1)
        return self.fc(x)


## Cell 11 — Deep Learning Detector: Ye-Net

Reimplementation of Ye, Ni & Yi (2017), *"Deep Learning Hierarchical
Representations for Image Steganalysis."* Key structural elements preserved:
30 fixed SRM high-pass filters (rather than Xu-Net's single KV filter) as the
preprocessing layer, a Truncated Linear Unit (TLU) activation immediately after
preprocessing (clips residuals to ±T, matching the truncation already used in
the SRM co-occurrence features elsewhere in this notebook), and a deeper
trainable conv stack than Xu-Net, reflecting the original paper's finding that
richer preprocessing filters allow — and require — more capacity downstream.


In [11]:
class TLU(nn.Module):
    """Truncated Linear Unit: clip(x, -T, T). Matches SRM residual truncation."""
    def __init__(self, T: float = 8.0):
        super().__init__()
        self.T = T

    def forward(self, x):
        return torch.clamp(x, -self.T, self.T)


def _build_30_srm_filters() -> torch.Tensor:
    """
    Builds a representative 30-filter SRM bank (5x5, zero-padded where the
    base kernel is smaller) for Ye-Net's fixed preprocessing layer. Uses the
    same first/second/third-order and edge kernels already defined for the
    classical SRM extractor (Cell 8), replicated/padded to a uniform 5x5 and
    padded out to 30 filters by including horizontally/vertically mirrored
    and rotated variants — consistent with the spirit of the original paper's
    filter bank without requiring the exact proprietary 30-filter set.
    """
    base = list(SRM_KERNELS.values())
    filters = []
    for k in base:
        kh, kw = k.shape
        padded = np.zeros((5, 5), dtype=np.float32)
        top, left = (5 - kh) // 2, (5 - kw) // 2
        padded[top:top+kh, left:left+kw] = k
        filters.append(padded)
        filters.append(np.rot90(padded, 1).copy())   # 90-degree rotation variant
    while len(filters) < 30:
        filters.append(np.rot90(filters[len(filters) % len(base)], 2).copy())
    filters = filters[:30]
    return torch.tensor(np.stack(filters), dtype=torch.float32).unsqueeze(1)  # (30,1,5,5)


class YeNet(nn.Module):
    """
    Ye-Net (Ye et al., 2017). Input: (B, 3, H, W).
    30 fixed SRM filters (per input channel) -> TLU -> 5 trainable conv blocks
    -> global average pool -> FC(2).
    """
    def __init__(self, in_channels: int = 3, tlu_threshold: float = 8.0):
        super().__init__()
        srm30 = _build_30_srm_filters()                          # (30,1,5,5)
        full_kernel = srm30.repeat(in_channels, 1, 1, 1)          # (30*C,1,5,5)
        self.register_buffer("srm30_kernel", full_kernel)
        self.in_channels = in_channels
        self.tlu = TLU(tlu_threshold)

        c0 = 30 * in_channels
        self.block1 = nn.Sequential(
            nn.Conv2d(c0, 30, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(30), nn.ReLU(True),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(30, 30, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(30), nn.ReLU(True),
            nn.AvgPool2d(3, stride=2, padding=1),
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(30, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(True),
            nn.AvgPool2d(3, stride=2, padding=1),
        )
        self.block4 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.AvgPool2d(3, stride=2, padding=1),
        )
        self.block5 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(True),
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, 2)

    def forward(self, x):
        x = F.conv2d(x, self.srm30_kernel, padding=2, groups=self.in_channels)
        x = self.tlu(x)
        x = self.block1(x); x = self.block2(x); x = self.block3(x)
        x = self.block4(x); x = self.block5(x)
        x = self.global_pool(x).flatten(1)
        return self.fc(x)


## Cell 12 — CNN Training & Evaluation Pipeline (shared by Xu-Net and Ye-Net)

One dataset class and one training/eval routine serve both models. Images are
kept as full 512×512×3 tensors (both networks are fully convolutional up to
the global average pool, so no patch-cropping is required, unlike some
steganalysis CNN pipelines that train on small patches for speed). Multiple
independent training trials (`CFG["n_trials_dl"]`) are run per payload rate to
report mean ± std, consistent with the SRM+LinearSVC protocol.


In [ ]:
class StegoDataset(Dataset):
    """Cover/stego pairs for a given payload rate, as (image, label) pairs."""
    def __init__(self, records: list, indices: np.ndarray):
        self.samples = []
        for idx in indices:
            self.samples.append((records[idx]["cover"], 0))
            self.samples.append((records[idx]["stego"], 1))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        img, label = self.samples[i]
        # BGR uint8 (H,W,3) -> float32 CHW tensor, scaled to [0,1]
        t = torch.from_numpy(img.astype(np.float32) / 255.0).permute(2, 0, 1)
        return t, label


def train_and_evaluate_cnn(
    model_cls,
    records: list,
    seed: int,
    epochs: int = CFG["dl_epochs"],
    batch_size: int = CFG["dl_batch_size"],
    lr: float = CFG["dl_lr"],
    val_split: float = CFG["dl_val_split"],
) -> dict:
    """
    One full training run: train/val split at the IMAGE level (not the
    cover/stego pair level, to avoid leaking a cover and its paired stego
    across the split), train for `epochs`, evaluate on the held-out split.
    Returns PE, accuracy, AUC, and ROC arrays, matching the SVM pipeline's
    return format so results can be compared directly.
    """
    rng = np.random.default_rng(seed)
    n_images = len(records)
    indices = rng.permutation(n_images)
    n_val = max(1, int(n_images * val_split))
    val_idx, train_idx = indices[:n_val], indices[n_val:]

    train_ds = StegoDataset(records, train_idx)
    val_ds   = StegoDataset(records, val_idx)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    model = model_cls().to(DEVICE)
    optimizer = torch.optim.Adamax(model.parameters(), lr=lr)   # Adamax, as in the original papers
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()

    model.eval()
    all_labels, all_scores, all_preds = [], [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(DEVICE)
            logits = model(imgs)
            probs  = F.softmax(logits, dim=1)[:, 1]   # P(stego)
            preds  = logits.argmax(dim=1).cpu().numpy()
            all_labels.extend(labels.numpy())
            all_scores.extend(probs.cpu().numpy())
            all_preds.extend(preds)

    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    pe  = 0.5 * (fp / (fp+tn+1e-12) + fn / (fn+tp+1e-12))
    acc = accuracy_score(all_labels, all_preds)
    fpr, tpr, _ = roc_curve(all_labels, all_scores)
    roc_auc = auc(fpr, tpr)

    return {"pe": pe, "acc": acc, "auc": roc_auc, "fpr": fpr, "tpr": tpr}


def run_cnn_benchmark(model_cls, model_name: str) -> dict:
    results = {bpp: [] for bpp in CFG["payload_rates"]}
    for bpp in CFG["payload_rates"]:
        print(f"\n{model_name}  bpp={bpp}")
        for trial in range(CFG["n_trials_dl"]):
            seed = CFG["seed"] + trial * 1000
            r = train_and_evaluate_cnn(model_cls, stego_bank[bpp], seed=seed)
            results[bpp].append(r)
            print(f"  trial {trial+1}/{CFG['n_trials_dl']}: "
                  f"PE={r['pe']:.4f}  Acc={r['acc']:.4f}  AUC={r['auc']:.4f}")
    return results


xunet_results = run_cnn_benchmark(XuNet, "Xu-Net")
yenet_results = run_cnn_benchmark(YeNet, "Ye-Net")


def aggregate_cnn_results(results: dict) -> dict:
    agg = {}
    for bpp, trials in results.items():
        agg[bpp] = {
            "pe_mean":  np.mean([t["pe"]  for t in trials]), "pe_std":  np.std([t["pe"]  for t in trials]),
            "auc_mean": np.mean([t["auc"] for t in trials]), "auc_std": np.std([t["auc"] for t in trials]),
        }
    return agg


xunet_aggregated = aggregate_cnn_results(xunet_results)
yenet_aggregated = aggregate_cnn_results(yenet_results)



Xu-Net  bpp=0.1


  trial 1/3: PE=0.3725  Acc=0.6275  AUC=0.7892
  trial 2/3: PE=0.3450  Acc=0.6550  AUC=0.7622
  trial 3/3: PE=0.2125  Acc=0.7875  AUC=0.8858

Xu-Net  bpp=0.2
  trial 1/3: PE=0.2775  Acc=0.7225  AUC=0.8279
  trial 2/3: PE=0.2775  Acc=0.7225  AUC=0.8144
  trial 3/3: PE=0.1875  Acc=0.8125  AUC=0.9229

Xu-Net  bpp=0.4
  trial 1/3: PE=0.2025  Acc=0.7975  AUC=0.9000
  trial 2/3: PE=0.2625  Acc=0.7375  AUC=0.8923
  trial 3/3: PE=0.4150  Acc=0.5850  AUC=0.7084

Ye-Net  bpp=0.1


## Cell 13 — Unified Detector Comparison (SRM+SVM vs. Xu-Net vs. Ye-Net)

In [ ]:
rows = []
for bpp in CFG["payload_rates"]:
    rows.append([bpp, "SRM + LinearSVC",
                 f"{svm_aggregated[bpp]['pe_mean']:.4f}±{svm_aggregated[bpp]['pe_std']:.4f}",
                 f"{svm_aggregated[bpp]['auc_mean']:.4f}±{svm_aggregated[bpp]['auc_std']:.4f}"])
    rows.append([bpp, "Xu-Net",
                 f"{xunet_aggregated[bpp]['pe_mean']:.4f}±{xunet_aggregated[bpp]['pe_std']:.4f}",
                 f"{xunet_aggregated[bpp]['auc_mean']:.4f}±{xunet_aggregated[bpp]['auc_std']:.4f}"])
    rows.append([bpp, "Ye-Net",
                 f"{yenet_aggregated[bpp]['pe_mean']:.4f}±{yenet_aggregated[bpp]['pe_std']:.4f}",
                 f"{yenet_aggregated[bpp]['auc_mean']:.4f}±{yenet_aggregated[bpp]['auc_std']:.4f}"])

print(tabulate(rows, headers=["Payload (bpp)", "Detector", "PE", "AUC"], tablefmt="grid"))

fig, axes = plt.subplots(1, len(CFG["payload_rates"]), figsize=(16, 5), sharey=True)
for ax, bpp in zip(axes, CFG["payload_rates"]):
    for name, trials, color in [
        ("SRM+SVM", svm_trial_results[bpp], "#2166ac"),
        ("Xu-Net",  xunet_results[bpp],      "#4dac26"),
        ("Ye-Net",  yenet_results[bpp],       "#d01c8b"),
    ]:
        base_fpr = np.linspace(0, 1, 200)
        tprs = [np.interp(base_fpr, t["fpr"], t["tpr"]) for t in trials]
        ax.plot(base_fpr, np.mean(tprs, axis=0), color=color, linewidth=2, label=name)
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")
    ax.set_title(f"{bpp} bpp"); ax.set_xlabel("FPR")
    if ax is axes[0]:
        ax.set_ylabel("TPR")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle(f"Detector Comparison — {CFG['dataset_name']}", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], "detector_comparison_roc.pdf"), dpi=300, bbox_inches="tight")
plt.savefig(os.path.join(CFG["output_dir"], "detector_comparison_roc.png"), dpi=200, bbox_inches="tight")
plt.show()


## Cell 14 — Unified Experimental Settings Table (all detectors)

In [ ]:
def build_experimental_table_v2() -> pd.DataFrame:
    rows = []
    for bpp in CFG["payload_rates"]:
        n_pairs = len(stego_bank[bpp])
        for name, agg in [
            ("SRM + LinearSVC", svm_aggregated[bpp]),
            ("Xu-Net",          xunet_aggregated[bpp]),
            ("Ye-Net",          yenet_aggregated[bpp]),
        ]:
            rows.append({
                "Dataset":              CFG["dataset_name"],
                "Resolution":           CFG["image_res"],
                "Detector":             name,
                "Embedding Rate (bpp)": bpp,
                "Samples (per class)":  n_pairs,
                "Trials":               CFG["n_trials"] if name == "SRM + LinearSVC" else CFG["n_trials_dl"],
                "PE (mean ± std)":      f"{agg['pe_mean']:.4f} ± {agg['pe_std']:.4f}",
                "AUC (mean ± std)":     f"{agg['auc_mean']:.4f} ± {agg['auc_std']:.4f}",
                "PSNR dB":              f"{quality_results[bpp]['psnr_mean']:.2f} ± {quality_results[bpp]['psnr_std']:.2f}",
                "SSIM":                 f"{quality_results[bpp]['ssim_mean']:.6f} ± {quality_results[bpp]['ssim_std']:.6f}",
            })
    return pd.DataFrame(rows)


exp_table = build_experimental_table_v2()
print(tabulate(exp_table, headers="keys", tablefmt="grid", showindex=False))

exp_table.to_csv(os.path.join(CFG["output_dir"], "experimental_table_v2.csv"), index=False)
with open(os.path.join(CFG["output_dir"], "experimental_table_v2.tex"), "w") as f:
    f.write(exp_table.to_latex(index=False, escape=True,
        caption="Detector comparison across embedding rates.", label="tab:detector_comparison"))
print("\nSaved CSV and LaTeX table.")


## Cell 15 — Unified Metrics

In [ ]:
def build_unified_metrics_table() -> pd.DataFrame:
    """
    Complete unified results table: every metric computed across the notebook
    (imperceptibility, classical statistical attacks, and all three ML/DL
    detectors), one row per (payload rate, detector), all in a single table.
    """
    rows = []
    for bpp in CFG["payload_rates"]:
        n_pairs = len(stego_bank[bpp])
        q  = quality_results[bpp]
        c2 = chi2_results[bpp]
        rs = rs_results[bpp]

        for name, agg in [
            ("SRM + LinearSVC", svm_aggregated[bpp]),
            ("Xu-Net",          xunet_aggregated[bpp]),
            ("Ye-Net",          yenet_aggregated[bpp]),
        ]:
            rows.append({
                "Dataset":               CFG["dataset_name"],
                "Resolution":            CFG["image_res"],
                "Detector":              name,
                "Embedding Rate (bpp)":  bpp,
                "Samples (per class)":   n_pairs,
                "Trials":                CFG["n_trials"] if name == "SRM + LinearSVC" else CFG["n_trials_dl"],

                # Imperceptibility
                "PSNR dB":               f"{q['psnr_mean']:.2f} ± {q['psnr_std']:.2f}",
                "SSIM":                  f"{q['ssim_mean']:.6f} ± {q['ssim_std']:.6f}",

                # Chi-square attack (cover and stego)
                "Chi² p (cover)":        f"{c2['cover_p_mean']:.4f}",
                "Chi² p (stego)":        f"{c2['stego_p_mean']:.4f}",

                # RS analysis (cover and stego)
                "RS Est. (cover)":       f"{rs['cover_mean']:.4f} ± {rs['cover_std']:.4f}",
                "RS Est. (stego)":       f"{rs['stego_mean']:.4f} ± {rs['stego_std']:.4f}",

                # Classifier / detector results
                "PE (mean ± std)":       f"{agg['pe_mean']:.4f} ± {agg['pe_std']:.4f}",
                "AUC (mean ± std)":      f"{agg['auc_mean']:.4f} ± {agg['auc_std']:.4f}",
            })

    return pd.DataFrame(rows)


unified_table = build_unified_metrics_table()

print("\n" + "=" * 120)
print("UNIFIED RESULTS — ALL METRICS, ALL DETECTORS, ALL PAYLOAD RATES")
print("=" * 120)
print(tabulate(unified_table, headers="keys", tablefmt="grid", showindex=False))

unified_table.to_csv(os.path.join(CFG["output_dir"], "unified_metrics_table.csv"), index=False)
with open(os.path.join(CFG["output_dir"], "unified_metrics_table.tex"), "w") as f:
    f.write(unified_table.to_latex(
        index=False, escape=True,
        caption="Complete steganalysis results: imperceptibility, classical statistical attacks, and ML/DL detector performance across all embedding rates.",
        label="tab:unified_metrics_full",
    ))
print(f"\nSaved to {CFG['output_dir']}/unified_metrics_table.csv and .tex")
